In [1]:
!pip install gradio ollama pillow numpy pandas

In [2]:
# =========================================================
# REMOTE OLLAMA DASHBOARD – Personalised UI Aesthetics
# =========================================================
import os, io, re, base64, numpy as np, pandas as pd
from PIL import Image
import gradio as gr
import ollama

In [ ]:
# ---------------------------
# 1. Remote Ollama settings
# ---------------------------
# Set your remote Ollama host and API key here (or as environment variables)
os.environ["OLLAMA_HOST"] = "https://ollama.com"   # ← replace with your server
os.environ["OLLAMA_API_KEY"] = "OLLAMA API KEY"   # ← replace with your key

# The exact model tag available on your remote server
OLLAMA_MODEL = "qwen3-vl:235b-cloud"   # change to e.g. "minicpm-v" or "llava-phi3"

# Create the client – it automatically reads OLLAMA_HOST & OLLAMA_API_KEY
ollama_client = ollama.Client()

# ---------------------------
# 2. Load statistics from the original CSV
# ---------------------------
CSV_PATH = "/kaggle/input/datasets/rahulsonboro/ui-aestheticdataset/Aes_dataset - Sheet1.csv"

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df.dropna(inplace=True)

# Per‑user mean rating
user_means = {}
for uid, group in df.groupby('user_id'):
    all_ratings = pd.concat([group['desktop_rating'], group['mobile_rating']])
    user_means[uid] = all_ratings.mean()

GLOBAL_MEAN = pd.concat([df['desktop_rating'], df['mobile_rating']]).mean()

# Client grouping (demographic)
def get_client_id(gender, qualification):
    gender_str = 'male' if gender == 0 else 'female'
    qual_map = {0: 'nonworking', 1: 'student', 2: 'working'}
    return f"{gender_str}_{qual_map[qualification]}"

user_client_map = {}
for _, row in df.iterrows():
    uid = row['user_id']
    if uid not in user_client_map:
        user_client_map[uid] = get_client_id(row['gender'], row['qualification'])

client_counts = {}
for uid, cid in user_client_map.items():
    client_counts[cid] = client_counts.get(cid, 0) + 1

# Average user_mean per demographic group
group_means = {}
for cid in client_counts:
    users_in_group = [uid for uid, c in user_client_map.items() if c == cid]
    group_means[cid] = np.mean([user_means[uid] for uid in users_in_group])

# ---------------------------
# 3. Prediction function
# ---------------------------
def predict_aesthetics(image, client_group):
    # Convert PIL image to base64 (Ollama expects this)
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

    # Map group ID to a human‑readable description
    demographic = {
        'male_student': 'male student',
        'female_student': 'female student',
        'male_working': 'male working professional',
        'female_working': 'female working professional',
        'female_nonworking': 'female non-working adult',
    }[client_group]

    # Craft the prompt
    prompt = (
        f"You are a {demographic}. "
        f"Rate the UI aesthetics from 1 (worst) to 10 (best). "
        f"Only reply with the number."
    )

    # Call remote Ollama
    response = ollama_client.chat(
        model=OLLAMA_MODEL,
        messages=[{
            "role": "user",
            "content": prompt,
            "images": [img_b64]
        }],
        options={"temperature": 0.0}     # deterministic
    )

    answer = response['message']['content'].strip()

    # Extract number 1‑10 from the answer
    try:
        numbers = re.findall(r'\b(10|[1-9])\b', answer)
        rating = float(numbers[-1]) if numbers else GLOBAL_MEAN
    except:
        rating = GLOBAL_MEAN
    return round(rating, 2)

# ---------------------------
# 4. Gradio Interface
# ---------------------------
client_options = list(group_means.keys())

iface = gr.Interface(
    fn=predict_aesthetics,
    inputs=[
        gr.Image(type="pil", label="Upload UI Screenshot"),
        gr.Dropdown(choices=client_options, value=client_options[0], label="Demographic Group")
    ],
    outputs=gr.Number(label="LLM Predicted Score (1–10)"),
    title="Ollama UI Aesthetic Predictor (Remote)",
    description="Select a demographic group. The remote vision model rates the UI from 1 to 10.",
    allow_flagging="never"
)

iface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2dbe7c7bac027cdb5d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
